# Containers Practice Notebook

Use this notebook to learn container and container-file operations step by step.

## 1. Setup

Before running:
1. Configure OCI credentials in `sandbox.yaml`.
2. Install project dependencies (`uv sync`).
3. Set test values in the next cell.

In [ ]:
from openai import OpenAI
from openai_sdk.openai_client_provider import OpenAIClientProvider

client: OpenAI = OpenAIClientProvider().oci_openai_client
print('Client ready')

In [ ]:
# Fill these values for your workspace
CONTAINER_ID = ''
FILE_ID = ''
LOCAL_FILE_PATH = 'data.csv'
DELETE_AT_END = False

## 2. Container Lifecycle

In [ ]:
created_container = client.containers.create(name='workshop-container')
print('Created:', created_container.id)
CONTAINER_ID = CONTAINER_ID or created_container.id

In [ ]:
containers_page = client.containers.list()
print('Containers returned:', len(containers_page.data))

In [ ]:
if not CONTAINER_ID:
    raise ValueError('Set CONTAINER_ID before retrieve')
container = client.containers.retrieve(CONTAINER_ID)
container

## 3. Container File Operations

In [ ]:
with open(LOCAL_FILE_PATH, 'rb') as fh:
    uploaded = client.containers.files.create(container_id=CONTAINER_ID, file=fh)
print('Uploaded file:', uploaded.id)
FILE_ID = FILE_ID or uploaded.id

In [ ]:
files_page = client.containers.files.list(container_id=CONTAINER_ID)
print('Files in container:', len(files_page.data))

In [ ]:
if not FILE_ID:
    raise ValueError('Set FILE_ID before retrieve/content')
file_meta = client.containers.files.retrieve(container_id=CONTAINER_ID, file_id=FILE_ID)
file_meta

In [ ]:
content = client.containers.files.content.retrieve(container_id=CONTAINER_ID, file_id=FILE_ID)
bytes_data = content.read()
print('Downloaded bytes:', len(bytes_data))

## 4. Optional Cleanup

In [ ]:
if DELETE_AT_END and FILE_ID:
    print(client.containers.files.delete(container_id=CONTAINER_ID, file_id=FILE_ID))
if DELETE_AT_END and CONTAINER_ID:
    print(client.containers.delete(CONTAINER_ID))
if not DELETE_AT_END:
    print('Skipping cleanup. Set DELETE_AT_END=True to delete resources.')